# Protein ↔ CellularComponent Relation-Wise Merge

Merges Protein–CellularComponent triples from CKG (×2), CrossBAR, and TARKG;
fills missing protein head names from UniProt; deduplicates by `(head, relation, tail)`;
and saves the result.

## 0. Configuration

In [1]:
import os
import pandas as pd
import numpy as np

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR  = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/PROTEIN_CELLULARCOMPONENT/ALL_PROTEIN_CELLULARCOMPONENT.parquet'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Protein Lookup Dictionary — UniProt

In [2]:
uniprot = pd.read_csv(DB_DIR + 'uniprot/uniprot_parsed_results.csv')
uniprot_trembl_AC_Name_dict = dict(zip(uniprot['AC'], uniprot['NAME']))
del uniprot
print(f"UniProt AC→Name dict: {len(uniprot_trembl_AC_Name_dict):,} entries")

UniProt AC→Name dict: 252,635,149 entries


## 2. Load KG Sources

### 2a. CKG (×2)

In [3]:
CKG_Protein_CC1 = pd.read_csv(PROC_DIR + 'CKG/File_20_protein_cellular_compartment_CKG.csv')
CKG_Protein_CC1.columns = CKG_Protein_CC1.columns.str.lower()
CKG_Protein_CC1['tail_id_is'] = 'Quick_GO'
CKG_Protein_CC1.rename(columns={'tail_go_name': 'tail_detail_name', 'head_alt_name': 'head_detail_name', 'kgsource': 'kg_source'}, inplace=True)

CKG_Protein_CC2 = pd.read_csv(PROC_DIR + 'CKG/File_14_Protein_Cellular_component_CKG.csv')
CKG_Protein_CC2.columns = CKG_Protein_CC2.columns.str.lower()
CKG_Protein_CC2['tail_id_is'] = 'Quick_GO'
CKG_Protein_CC2.rename(columns={'tail_go_name': 'tail_detail_name', 'head_alt_name': 'head_detail_name', 'kgsource': 'kg_source'}, inplace=True)

print(f"CKG 1: {len(CKG_Protein_CC1):,} rows")
print(f"CKG 2: {len(CKG_Protein_CC2):,} rows")

CKG 1: 20,555,035 rows
CKG 2: 288,342 rows


In [4]:
CKG_Protein_CC1['kg_type'] = 'Generalised'
CKG_Protein_CC1['kg_source'] = 'CKG'
CKG_Protein_CC1['species'] = 'Homo species'
CKG_Protein_CC1
CKG_Protein_CC1

,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,evidence_type,score,kg_type,species
0,Q9NRF2,ASSOCIATED_WITH_INTEGRATED,GO:1902495,Protein,Cellular_component,COMPARTMENTS,CKG,SH2B adapter protein 1,transmembrane transporter complex,Uniprot_ID,Quick_GO,compiled,0.602,Generalised,Homo species
1,Q9HB63,ASSOCIATED_WITH_INTEGRATED,GO:0098590,Protein,Cellular_component,COMPARTMENTS,CKG,Netrin-4,plasma membrane region,Uniprot_ID,Quick_GO,compiled,0.560,Generalised,Homo species
2,B1AM41,ASSOCIATED_WITH_INTEGRATED,GO:0030424,Protein,Cellular_component,COMPARTMENTS,CKG,Tumor necrosis factor receptor superfamily mem...,axon,Uniprot_ID,Quick_GO,compiled,2.001,Generalised,Homo species
3,B7Z8R3,ASSOCIATED_WITH_INTEGRATED,GO:0043540,Protein,Cellular_component,COMPARTMENTS,CKG,"Hydroxymethylglutaryl-CoA synthase, mitochondrial","6-phosphofructo-2-kinase/fructose-2,6-biphosph...",Uniprot_ID,Quick_GO,compiled,0.810,Generalised,Homo species
4,Q9H7B5,ASSOCIATED_WITH_INTEGRATED,GO:0110165,Protein,Cellular_component,COMPARTMENTS,CKG,Opioid growth factor receptor-like protein 1,cellular anatomical structure,Uniprot_ID,Quick_GO,compiled,3.398,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20555030,Q9UFF1,ASSOCIATED_WITH_INTEGRATED,GO:0009295,Protein,Cellular_component,COMPARTMENTS,CKG,PDZ domain-containing protein 8 {ECO:0000305},nucleoid,Uniprot_ID,Quick_GO,compiled,1.349,Generalised,Homo species
20555031,Q68DW1,ASSOCIATED_WITH_INTEGRATED,GO:0005575,Protein,Cellular_component,COMPARTMENTS,CKG,Protein phosphatase 1E,cellular_component,Uniprot_ID,Quick_GO,compiled,5.000,Generalised,Homo species
20555032,Q86VP9,ASSOCIATED_WITH_INTEGRATED,GO:0005740,Protein,Cellular_component,COMPARTMENTS,CKG,Centrosome-associated protein ALMS1 {ECO:0000305},mitochondrial envelope,Uniprot_ID,Quick_GO,compiled,0.810,Generalised,Homo species
20555033,P35611,ASSOCIATED_WITH_INTEGRATED,GO:1903561,Protein,Cellular_component,COMPARTMENTS,CKG,Alpha-adducin,extracellular vesicle,Uniprot_ID,Quick_GO,compiled,1.017,Generalised,Homo species


In [5]:
CKG_Protein_CC2['kg_type'] = 'Generalised'
CKG_Protein_CC2['kg_source'] = 'CKG'
CKG_Protein_CC2['species'] = 'Homo species'
CKG_Protein_CC2


,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,score,kg_type,species
0,P55854,ASSOCIATED_WITH,GO:0005654,Protein,Cellular_component,UniProt,CKG,Small ubiquitin-related modifier 3 {ECO:0000305},nucleoplasm,Uniprot_ID,Quick_GO,5,Generalised,Homo species
1,P32927,ASSOCIATED_WITH,GO:0009897,Protein,Cellular_component,UniProt,CKG,Cytokine receptor common subunit beta,external side of plasma membrane,Uniprot_ID,Quick_GO,5,Generalised,Homo species
2,Q9UBF8,ASSOCIATED_WITH,GO:0000139,Protein,Cellular_component,UniProt,CKG,Phosphatidylinositol 4-kinase beta {ECO:0000305},Golgi membrane,Uniprot_ID,Quick_GO,5,Generalised,Homo species
3,A8TX70,ASSOCIATED_WITH,GO:0005576,Protein,Cellular_component,UniProt,CKG,Collagen alpha-5(VI) chain,extracellular region,Uniprot_ID,Quick_GO,5,Generalised,Homo species
4,Q9GZZ6,ASSOCIATED_WITH,GO:0043005,Protein,Cellular_component,UniProt,CKG,Neuronal acetylcholine receptor subunit alpha-10,neuron projection,Uniprot_ID,Quick_GO,5,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
288337,A4GXA9,ASSOCIATED_WITH,GO:0048476,Protein,Cellular_component,UniProt,CKG,Probable crossover junction endonuclease EME2,Holliday junction resolvase complex,Uniprot_ID,Quick_GO,5,Generalised,Homo species
288338,P16066,ASSOCIATED_WITH,GO:0005886,Protein,Cellular_component,UniProt,CKG,Atrial natriuretic peptide receptor 1 {ECO:000...,plasma membrane,Uniprot_ID,Quick_GO,5,Generalised,Homo species
288339,P49754,ASSOCIATED_WITH,GO:0016020,Protein,Cellular_component,UniProt,CKG,Vacuolar protein sorting-associated protein 41...,membrane,Uniprot_ID,Quick_GO,5,Generalised,Homo species
288340,O43464,ASSOCIATED_WITH,GO:0032991,Protein,Cellular_component,UniProt,CKG,"Serine protease HTRA2, mitochondrial",protein-containing complex,Uniprot_ID,Quick_GO,5,Generalised,Homo species


### 2b. CrossBAR

In [6]:
CrossBAR_Protein_CC = pd.read_csv(PROC_DIR + 'crossbar/Protein_CellularComponent_Crossbar.csv')
CrossBAR_Protein_CC.columns = CrossBAR_Protein_CC.columns.str.lower()
CrossBAR_Protein_CC['tail_id_is'] = 'Quick_GO'
print(f"CrossBAR: {len(CrossBAR_Protein_CC):,} rows")
CrossBAR_Protein_CC['kg_type'] = 'Generalised'
CrossBAR_Protein_CC['kg_source'] = 'crossbar'
CrossBAR_Protein_CC['species'] = 'Homo species'
CrossBAR_Protein_CC
CrossBAR_Protein_CC.head(2)

CrossBAR: 768,522 rows


,head,relation,tail,head_type,tail_type,kg_source,head_alt_id,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,A1DG68,Protein_CellularComponent,GO:0005634,Protein,CellularComponent,crossbar,Leucine aminopeptidase 2,nucleus,Uniprot_ID,Quick_GO,Generalised,Homo species
1,A1DG68,Protein_CellularComponent,GO:0005737,Protein,CellularComponent,crossbar,Leucine aminopeptidase 2,cytoplasm,Uniprot_ID,Quick_GO,Generalised,Homo species


### 2c. TARKG

In [7]:
TARKG_Protein_CC = pd.read_csv(PROC_DIR + 'TARKG/Protein_CellularComponent_TARKG.csv')
TARKG_Protein_CC.columns = TARKG_Protein_CC.columns.str.lower()
print(f"TARKG:    {len(TARKG_Protein_CC):,} rows")
TARKG_Protein_CC['kg_type'] = 'Generalised'
TARKG_Protein_CC['kg_source'] = 'TARKG'
TARKG_Protein_CC['species'] = 'Homo species'
TARKG_Protein_CC.head(2)

TARKG:    325,394 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,P54198,Protein_Cellular Component,GO:0000775,Protein,GO_CC,Cellular Component,TARKG,Protein HIRA,"chromosome, centromeric region",Uniprot_ID,Quick_GO,BioKG-3222152,BioKG,0,Generalised,Homo species
1,E9PVX6,Protein_Cellular Component,GO:0000775,Protein,GO_CC,Cellular Component,TARKG,Proliferation marker protein Ki-67 {ECO:0000305},"chromosome, centromeric region",Uniprot_ID,Quick_GO,BioKG-3244149,BioKG,0,Generalised,Homo species


## 3. Check and Fix Duplicate Columns

In [8]:
SOURCE_DFS = [
    ('CKG_Protein_CC1',     CKG_Protein_CC1),
    ('CKG_Protein_CC2',     CKG_Protein_CC2),
    ('CrossBAR_Protein_CC', CrossBAR_Protein_CC),
    ('TARKG_Protein_CC',    TARKG_Protein_CC),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[CKG_Protein_CC1] ✓ no duplicates
[CKG_Protein_CC2] ✓ no duplicates
[CrossBAR_Protein_CC] ✓ no duplicates
[TARKG_Protein_CC] ✓ no duplicates


## 4. Align Schemas and Concatenate

In [9]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 21,937,293 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,Q9NRF2,ASSOCIATED_WITH_INTEGRATED,GO:1902495,Protein,NaN,Cellular_component,CKG,Uniprot_ID,Quick_GO,SH2B adapter protein 1,transmembrane transporter complex,Generalised,Homo species
1,Q9HB63,ASSOCIATED_WITH_INTEGRATED,GO:0098590,Protein,NaN,Cellular_component,CKG,Uniprot_ID,Quick_GO,Netrin-4,plasma membrane region,Generalised,Homo species


## 5. Standardise Fixed-Value Columns

In [10]:
final_df['head_type'] = 'Protein'
final_df['tail_type'] = 'CellularComponent'
final_df['relation']  = 'Protein_CellularComponent'

print("Unique relation:",  set(final_df['relation']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'Protein_CellularComponent'}
Unique tail_id_is: {'Quick_GO'}
Unique kg_source: {'TARKG', 'crossbar', 'CKG'}


## 6. Deduplicate

In [11]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 21,220,680 rows


## 7. Fill Missing Protein Head Names from UniProt



In [12]:
# Normalise fake NaNs to real np.nan
final_df_dedup['head_detail_name'] = final_df_dedup['head_detail_name'].replace(['NAN', 'NaN', 'None'], np.nan)

# Fill from UniProt AC→Name dict
final_df_dedup['head_detail_name'] = final_df_dedup['head_detail_name'].fillna(
    final_df_dedup['head'].map(uniprot_trembl_AC_Name_dict)
)

# Drop rows still missing a head name
before = len(final_df_dedup)
final_df_dedup = final_df_dedup[~final_df_dedup['head_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df_dedup):,} rows with missing head_detail_name → {len(final_df_dedup):,} remaining")

Dropped 436,915 rows with missing head_detail_name → 20,783,765 remaining


## 8. Add Schema Columns and Enforce Column Order

In [13]:

final_df_dedup = final_df_dedup[REQUIRED_COLS]
print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Final shape: (20783765, 13)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,A0A024R072,Protein_CellularComponent,GO:0000015,Protein,None,CellularComponent,CKG,Uniprot_ID,Quick_GO,Cyclin-L2,phosphopyruvate hydratase complex,Generalised,Homo species
1,A0A024R072,Protein_CellularComponent,GO:0000123,Protein,None,CellularComponent,CKG,Uniprot_ID,Quick_GO,Cyclin-L2,histone acetyltransferase complex,Generalised,Homo species
2,A0A024R072,Protein_CellularComponent,GO:0000124,Protein,None,CellularComponent,CKG,Uniprot_ID,Quick_GO,Cyclin-L2,SAGA complex,Generalised,Homo species
3,A0A024R072,Protein_CellularComponent,GO:0000133,Protein,None,CellularComponent,CKG,Uniprot_ID,Quick_GO,Cyclin-L2,polarisome,Generalised,Homo species
4,A0A024R072,Protein_CellularComponent,GO:0000139,Protein,None,CellularComponent,CKG,Uniprot_ID,Quick_GO,Cyclin-L2,Golgi membrane,Generalised,Homo species


## 9. QC — NaN Check

In [14]:
final_df_dedup['species'] = 'Homo sapiens'

In [15]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,20458371,0,20458371
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 10. Save Output

In [16]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_parquet(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 20,783,765 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PROTEIN_CELLULARCOMPONENT/ALL_PROTEIN_CELLULARCOMPONENT.parquet


In [18]:
#